## Parte 3 · YAML de configuración 


Crea desde cero un fichero analisis.yaml que configure tu propio proceso: rutas de los ficheros de entrada, campos obligatorios, importe mínimo para incluir una licitación y formato de salida.

1. Escríbelo con comentarios que expliquen cada bloque.

2. Cárgalo con yaml.safe_load() y usa realmente esos valores en el resto del cuaderno. No debe quedar ninguna ruta ni umbral escrito a mano en el código.


3. Explica por qué se usa safe_load y no load.


4. Incluye a propósito un valor que YAML interpretaría mal sin comillas (por ejemplo no o 1.10), enséñalo y corrígelo.

### 3.1. Fichero `analisis.yaml`



Se crea desde cero con un comentario por bloque:

```yaml
# ============================================================
# Configuración del proceso de análisis de licitaciones
# ============================================================

# --- Entradas: rutas de los ficheros que se van a procesar ---
entradas:
  # Fichero JSON generado por la API (parte 1)
  json: "datos/licitaciones_api.json"
  # Fichero XML del canal Atom de la PCSP (parte 2)
  xml: "datos/licitaciones_pcsp.xml"

# --- Validación: reglas que debe cumplir cada licitación ---
validacion:
  # Campos que debe tener toda licitación para procesarse
  campos_obligatorios:
    - id
    - organo
    - objeto
    - importe
    - fechaLimite
  # Importe mínimo (EUR) para incluir una licitación en el resultado
  importe_minimo: 100000.0
  # Si es true, se rechazan importes <= 0 aunque no haya mínimo
  exigir_importe_positivo: true

# --- Salida: formato y ruta del resultado ---
salida:
  # Formato del fichero de salida: csv, json o markdown
  formato: "csv"
  # Ruta donde se escribe el resultado
  ruta: "resultados/licitaciones_filtradas.csv"
  # Codificación del fichero de salida
  encoding: "utf-8"
  # Versión del formato de salida: se escribe como cadena para conservar el 1.10
  version_formato: "1.10"

### 3.2. Carga con `yaml.safe_load()` y uso real de los valores

Se carga el fichero una sola vez y se guardan los valores en variables. A partir de este punto, **ninguna ruta ni umbral se escribe a mano** en el código.

```python
import yaml

with open("analisis.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Entradas
RUTA_JSON = config["entradas"]["json"]
RUTA_XML  = config["entradas"]["xml"]

# Validación
OBLIGATORIOS        = config["validacion"]["campos_obligatorios"]
IMPORTE_MIN         = config["validacion"]["importe_minimo"]
EXIGIR_POSITIVO     = config["validacion"]["exigir_importe_positivo"]

# Salida
FORMATO        = config["salida"]["formato"]
RUTA_SALIDA    = config["salida"]["ruta"]
ENCODING       = config["salida"]["encoding"]
VERSION_FORMATO = config["salida"]["version_formato"]

# Comprobación rápida
print(RUTA_JSON)        # datos/licitaciones_api.json
print(IMPORTE_MIN)      # 100000.0
print(OBLIGATORIOS)     # ['id', 'organo', 'objeto', 'importe', 'fechaLimite']
print(VERSION_FORMATO)  # 1.10  (str, gracias a las comillas)

### 3.3. ¿Por qué `yaml.safe_load()` y no `yaml.load()`?

`yaml.load()` **construye objetos Python arbitrarios** a partir del YAML. Eso incluye etiquetas especiales como `!!python/object/apply`, que pueden **ejecutar código** al cargar el fichero. Ejemplo peligroso:

```yaml
malicioso: !!python/object/apply:os.system ["echo hola > /tmp/pwned"]

### 3.4. Valor que YAML interpretaría mal sin comillas

YAML no sabe qué tipo queremos: si no se entrecomilla, deduce el tipo a partir de la forma del valor. Dos casos típicos en este fichero:

| Valor sin comillas | YAML lo interpreta como | Resultado |
|---|---|---|
| `no` | booleano (YAML 1.1) | `False` |
| `1.10` | número decimal | `1.1` (float) |

#### Ejemplo con `no`

Versión **incorrecta**:

```yaml
salida:
  incluir_cabecera: no